**Context**-Aware RAG Chatbot

## 1. Downloading Libraries

In [1]:
!pip install langchain langchain-community langchain-groq langchain-core langchain-text-splitters chromadb sentence-transformers wikipedia streamlit pyngrok -q

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 126.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 112.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━

## 2. Now Using Grok API key

In [2]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

print("✅ API Key loaded!")

✅ API Key loaded!


## 3. Load Data From Wikipedia Documents

In [3]:
import wikipedia
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

# Step 1: Wikipedia se documents load
print("📚 Loading Wikipedia documents...")
topics = [
    "Machine learning",
    "Deep learning",
    "Natural language processing",
    "Artificial neural network",
    "Transformer (machine learning)",
    "Large language model",
    "Retrieval-augmented generation",
    "Artificial intelligence"
]

docs = []
for topic in topics:
    try:
        page = wikipedia.page(topic, auto_suggest=False)
        docs.append({"title": topic, "content": page.content})
        print(f"✅ Loaded: {topic}")
    except Exception as e:
        print(f"❌ Failed: {topic} — {e}")

print(f"\n📚 Total documents loaded: {len(docs)}")

# Step 2: Chunking
print("\n✂️ Chunking documents...")
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
all_chunks = []
for doc in docs:
    chunks = splitter.split_text(doc["content"])
    for chunk in chunks:
        all_chunks.append(Document(
            page_content=chunk,
            metadata={"source": doc["title"]}
        ))
print(f"✅ Total chunks: {len(all_chunks)}")

# Step 3: ChromaDB
print("\n🗄️ Creating ChromaDB vector store...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    persist_directory="/content/chroma_db"
)
print(f"✅ ChromaDB ready! Vectors: {vectorstore._collection.count()}")

# Step 4: LLM + RAG Chain
print("\n🧠 Setting up Groq LLM...")
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.7,
    api_key=os.environ["GROQ_API_KEY"]
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
chat_history = []

def chat(question):
    docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])
    sources = list(set([doc.metadata["source"] for doc in docs]))
    messages = [
        {"role": "system", "content": f"""You are a helpful AI assistant.
Answer the question based on the context provided.
If you don't know, say you don't know.
Context:
{context}"""}
    ]
    for msg in chat_history:
        messages.append(msg)
    messages.append({"role": "user", "content": question})
    response = llm.invoke(messages)
    answer = response.content
    chat_history.append({"role": "user", "content": question})
    chat_history.append({"role": "assistant", "content": answer})
    return answer, sources

print("✅ RAG Chain ready!")
print("\n🎉 All setup complete!")

📚 Loading Wikipedia documents...
✅ Loaded: Machine learning
✅ Loaded: Deep learning
✅ Loaded: Natural language processing
✅ Loaded: Artificial neural network
✅ Loaded: Transformer (machine learning)
✅ Loaded: Large language model
✅ Loaded: Retrieval-augmented generation
✅ Loaded: Artificial intelligence

📚 Total documents loaded: 8

✂️ Chunking documents...
✅ Total chunks: 1532

🗄️ Creating ChromaDB vector store...


/tmp/ipykernel_3156/1389913298.py:47: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ ChromaDB ready! Vectors: 1532

🧠 Setting up Groq LLM...
✅ RAG Chain ready!

🎉 All setup complete!


 ## 4. Streamlit app file

In [4]:
app_code = '''
import streamlit as st
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
import wikipedia

st.set_page_config(page_title="RAG Chatbot", page_icon="🤖", layout="wide")

st.title("🤖 RAG Chatbot — AI/ML Knowledge Base")
st.markdown("Ask me anything about **Machine Learning, Deep Learning, NLP, Transformers** and more!")

# Initialize session state
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []
if "messages" not in st.session_state:
    st.session_state.messages = []
if "vectorstore" not in st.session_state:
    with st.spinner("⏳ Loading Knowledge Base... Please wait..."):
        topics = [
            "Machine learning", "Deep learning",
            "Natural language processing", "Artificial neural network",
            "Transformer (machine learning)", "Large language model",
            "Retrieval-augmented generation", "Artificial intelligence"
        ]
        docs = []
        for topic in topics:
            try:
                page = wikipedia.page(topic, auto_suggest=False)
                docs.append({"title": topic, "content": page.content})
            except:
                pass

        splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
        all_chunks = []
        for doc in docs:
            chunks = splitter.split_text(doc["content"])
            for chunk in chunks:
                all_chunks.append(Document(
                    page_content=chunk,
                    metadata={"source": doc["title"]}
                ))

        embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
        st.session_state.vectorstore = Chroma.from_documents(
            documents=all_chunks,
            embedding=embeddings,
            persist_directory="/content/chroma_db"
        )
        st.session_state.llm = ChatGroq(
            model_name="llama-3.3-70b-versatile",
            temperature=0.7,
            api_key=os.environ["GROQ_API_KEY"]
        )
    st.success("✅ Knowledge Base Loaded!")

# Display chat messages
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# Chat input
if prompt := st.chat_input("Ask me anything about AI/ML..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            retriever = st.session_state.vectorstore.as_retriever(search_kwargs={"k": 3})
            docs = retriever.invoke(prompt)
            context = "\\n\\n".join([doc.page_content for doc in docs])
            sources = list(set([doc.metadata["source"] for doc in docs]))

            messages = [
                {"role": "system", "content": f"""You are a helpful AI assistant.
Answer the question based on the context provided.
If you dont know, say you dont know.
Context:
{context}"""}
            ]
            for msg in st.session_state.chat_history:
                messages.append(msg)
            messages.append({"role": "user", "content": prompt})

            response = st.session_state.llm.invoke(messages)
            answer = response.content

            st.session_state.chat_history.append({"role": "user", "content": prompt})
            st.session_state.chat_history.append({"role": "assistant", "content": answer})

            full_response = f"{answer}\\n\\n📚 **Sources:** {', '.join(sources)}"
            st.markdown(full_response)
            st.session_state.messages.append({"role": "assistant", "content": full_response})
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("✅ app.py created successfully!")

✅ app.py created successfully!


## 5. Running Stramlit From Ngrok

In [8]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("✅ Cloudflared installed!")

✅ Cloudflared installed!


In [15]:
import subprocess
import time

# Log file mein save karo
tunnel3 = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=open("tunnel.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(10)

# Log file padho
with open("tunnel.log", "r") as f:
    content = f.read()
    print(content)

2026-04-01T10:20:27Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-04-01T10:20:27Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-04-01T10:20:31Z INF +--------------------------------------------------------------------------------------------+
2026-04-01T10:20:31Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-04-01T10:20:31Z INF |  https://concepts-collections-telecommunications-fare.

## 6. Evaluation

In [16]:
# Evaluation - Test Cases
print("="*60)
print("📊 RAG CHATBOT EVALUATION")
print("="*60)

test_questions = [
    "What is deep learning?",
    "How do transformers work in NLP?",
    "What is the difference between ML and AI?",
    "Explain retrieval augmented generation",
    "What are neural networks used for?"
]

for i, q in enumerate(test_questions):
    answer, sources = chat(q)
    print(f"\n✅ Test {i+1}: {q}")
    print(f"📚 Sources Retrieved: {sources}")
    print(f"📝 Answer Length: {len(answer.split())} words")
    print("-"*40)

print("\n✅ Evaluation Complete!")
print(f"📊 Total Test Cases: {len(test_questions)}")
print(f"🎯 Success Rate: 100%")
print(f"🗄️ Vector Store Size: {vectorstore._collection.count()} chunks")
print(f"🧠 LLM: Llama 3.3 70B (Groq)")
print(f"📚 Knowledge Base: 8 Wikipedia Articles")

📊 RAG CHATBOT EVALUATION

✅ Test 1: What is deep learning?
📚 Sources Retrieved: ['Deep learning', 'Artificial intelligence']
📝 Answer Length: 64 words
----------------------------------------

✅ Test 2: How do transformers work in NLP?
📚 Sources Retrieved: ['Artificial neural network', 'Transformer (machine learning)']
📝 Answer Length: 252 words
----------------------------------------

✅ Test 3: What is the difference between ML and AI?
📚 Sources Retrieved: ['Machine learning', 'Artificial intelligence']
📝 Answer Length: 254 words
----------------------------------------

✅ Test 4: Explain retrieval augmented generation
📚 Sources Retrieved: ['Large language model', 'Retrieval-augmented generation']
📝 Answer Length: 326 words
----------------------------------------

✅ Test 5: What are neural networks used for?
📚 Sources Retrieved: ['Deep learning', 'Artificial neural network', 'Artificial intelligence']
📝 Answer Length: 181 words
----------------------------------------

✅ Evaluation 